# 02b – Agregação semanal e unificação das séries

**Entradas:** `data/raw/casos_covid_parana_raw.csv`, `data/raw/checagens_lupa_raw.csv`  
**Saída:** `data/processed/series_semanais_curitiba.csv` (208 semanas, 2020–2023)

- Casos: filtro `MUN_RESIDENCIA == CURITIBA`, data `DATA_DIAGNOSTICO`, soma semanal.
- Checagens: data `data_publicacao`, contagem semanal (sem filtro geográfico, abrangência nacional).
- Semanas: `resample('W-SUN')`, ou seja, segunda a domingo, rotuladas pelo domingo.
- O recorte 2020–2022 (155 semanas) é aplicado no notebook 03.

In [1]:
# 1. Importar bibliotecas essenciais
import pandas as pd
import os # Para verificar se arquivos existem

print("--- Iniciando Notebook 02b: Agregação Semanal ---")

# 2. Recalcular 'casos_por_dia' a partir dos dados brutos de COVID
path_covid_raw = '../data/raw/casos_covid_parana_raw.csv'

if not os.path.exists(path_covid_raw):
    print(f"ERRO: Arquivo de dados brutos COVID '{path_covid_raw}' não encontrado.")
else:
    try:
        print(f"Carregando dados brutos COVID de: {path_covid_raw}")
        df_covid = pd.read_csv(path_covid_raw, sep=';', encoding='latin1')

        # Filtrar Curitiba
        df_curitiba = df_covid[df_covid['MUN_RESIDENCIA'].str.upper() == 'CURITIBA'].copy()

        # Tratar datas
        df_curitiba['DATA_DIAGNOSTICO'] = pd.to_datetime(df_curitiba['DATA_DIAGNOSTICO'], errors='coerce')
        df_curitiba.dropna(subset=['DATA_DIAGNOSTICO'], inplace=True) # Remover datas inválidas

        # Agrupar por dia
        casos_por_dia = df_curitiba.groupby('DATA_DIAGNOSTICO').size()
        casos_por_dia.index = pd.to_datetime(casos_por_dia.index) # Garantir que o índice é DatetimeIndex
        print("Série 'casos_por_dia' recalculada com sucesso.")

    except Exception as e:
        print(f"Erro ao processar dados COVID: {e}")
        casos_por_dia = None # Define como None se houver erro

# 3. Carregar dados brutos da Lupa e Agregar Semanalmente
path_lupa_raw = '../data/raw/checagens_lupa_raw.csv'
df_final_semanal = None # Inicializa como None

if not os.path.exists(path_lupa_raw):
    print(f"ERRO: Arquivo de dados brutos Lupa '{path_lupa_raw}' não encontrado. Execute o Notebook 01.")
elif casos_por_dia is None:
     print("Não foi possível continuar sem a série 'casos_por_dia'. Verifique erros anteriores.")
else:
    try:
        print(f"Carregando dados brutos Lupa de: {path_lupa_raw}")
        df_lupa = pd.read_csv(path_lupa_raw, encoding='utf-8')

        # Converter data da Lupa
        df_lupa['data_publicacao'] = pd.to_datetime(df_lupa['data_publicacao'], errors='coerce')
        df_lupa.dropna(subset=['data_publicacao'], inplace=True)

        # --- Agregação Semanal ---
        # Agregar casos COVID (já temos 'casos_por_dia')
        casos_semanais = casos_por_dia.resample('W-SUN').sum()
        casos_semanais.name = 'casos_covid'
        print(f"Série de casos COVID agregada semanalmente: {casos_semanais.shape[0]} semanas.")

        # Agregar checagens Lupa
        df_lupa_temp = df_lupa.set_index('data_publicacao')
        desinfo_semanal = df_lupa_temp.resample('W-SUN').size()
        desinfo_semanal.name = 'checagens_lupa'
        print(f"Série de desinformação agregada semanalmente: {desinfo_semanal.shape[0]} semanas.")
        desinfo_semanal.index = desinfo_semanal.index.tz_localize(None)

        # --- Unificação e Limpeza ---
        df_final_semanal = pd.merge(casos_semanais, desinfo_semanal, left_index=True, right_index=True, how='outer')
        df_final_semanal['checagens_lupa'] = df_final_semanal['checagens_lupa'].fillna(0).astype(int)
        df_final_semanal['casos_covid'] = df_final_semanal['casos_covid'].fillna(0).astype(int)

        # Filtrar para o período de interesse (2020-2023)
        df_final_semanal = df_final_semanal.loc['2020-01-01':'2023-12-31']

        print("\nDataFrame Semanal Final (primeiras 5 linhas):")
        print(df_final_semanal.head())
        print(f"\nNúmero total de semanas no DataFrame final: {df_final_semanal.shape[0]}")

        # --- Salvar Resultado ---
        path_processed = '../data/processed/series_semanais_curitiba.csv'
        df_final_semanal.to_csv(path_processed, index=True)
        print(f"\nDataFrame semanal final salvo com sucesso em: {path_processed}")

    except Exception as e:
        print(f"Ocorreu um erro inesperado durante a agregação Lupa/Unificação: {e}")

# 4. Fim do Notebook
if df_final_semanal is not None and not df_final_semanal.empty:
    print("\n--- Notebook 02b concluído com sucesso! ---")
else:
    print("\n--- Notebook 02b concluído com erros. Verifique as mensagens acima. ---")

--- Iniciando Notebook 02b: Agregação Semanal ---
Carregando dados brutos COVID de: ../data/raw/casos_covid_parana_raw.csv


C:\Users\Henrique\AppData\Local\Temp\ipykernel_10932\587371216.py:15: DtypeWarning: Columns (12,13,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_covid = pd.read_csv(path_covid_raw, sep=';', encoding='latin1')


Série 'casos_por_dia' recalculada com sucesso.
Carregando dados brutos Lupa de: ../data/raw/checagens_lupa_raw.csv
Série de casos COVID agregada semanalmente: 203 semanas.
Série de desinformação agregada semanalmente: 207 semanas.

DataFrame Semanal Final (primeiras 5 linhas):
            casos_covid  checagens_lupa
2020-01-12            0               1
2020-01-19            0               3
2020-01-26            0               2
2020-02-02            0               7
2020-02-09            0               3

Número total de semanas no DataFrame final: 208

DataFrame semanal final salvo com sucesso em: ../data/processed/series_semanais_curitiba.csv

--- Notebook 02b concluído com sucesso! ---
